# Deep Learning to detect writed numbers

## Librairies

In [2]:
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from nn import transform
import os


In [7]:
batch_size = 16
n_epochs = 10
lr = 0.001
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
device

device(type='cpu')

## Dataset

In [4]:
dataset_path = 'dataset/'
classes = range(len(os.listdir(dataset_path)))

In [9]:
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
n_classes = len(dataset.classes)

FileNotFoundError: Found no valid file for the classes 0, 1, 2, 3, 4, 5, 6, 7, 8, 9. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [ ]:
#Function to show an image
def imshow(img, title):
    img = img / 2 + 0.5 #unormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)), cmap='gray')
    plt.title(title)
    plt.show()

#Get some random training images
dataiter = iter(dataloader)
images, labels = next(dataiter)

#Show images with labels
for i in range(len(images)):
    print(images[i].shape, labels[i].item())
    imshow(images[i],title=f'Label : {labels[i].item()}')

In [3]:
#total number of images
print(len(dataset))

NameError: name 'dataset' is not defined

In [6]:
n_classes

NameError: name 'n_classes' is not defined

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(classes, [len(os.listdir(dataset_path + c)) for c in os.listdir(dataset_path)])
plt.xticks(classes)
plt.title('Number of images per class')
plt.xlabel('Classes')
plt.ylabel('Number of images')
plt.show()

## Neural Network

In [ ]:
from nn import Net
net = Net(n_classes=n_classes).to(device)

In [ ]:
net

In [ ]:
#number of parameters
sum(p.numel() for p in net.parameters())

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)

#Train the network
for epoch in range(n_epochs): #loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(dataloader):
        #get the inputs, data is a list of [inputs, labels]
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = net(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    #print loss
    print(f'Epoch {epoch +1}, loss : {running_loss / len(dataloader)}')

print('Finished Training')

In [ ]:
#Save the model
torch.save(net.state_dict(), 'net.pth')

## Inference

### Total accuracy and accuracy per class

In [ ]:
test_dataset_path = 'test_dataset/'
test_dataset = datasets.ImageFolder(root=test_dataset_path, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
n_classes = len(test_dataset.classes)
classes = range(len(os.listdir(test_dataset_path)))

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(classes, [len(os.listdir(test_dataset_path + c)) for c in os.listdir(test_dataset_path)])
plt.xticks(classes)
plt.title('Number of images per class')
plt.xlabel('Classes')
plt.ylabel('Number of images')
plt.show()

In [ ]:
from nn import Net
net = Net(n_classes=10).to(device)
net.load_state_dict(torch.load('net.pth'))

In [ ]:
#compute accuracy on this test set, and compute accuracy per class and plot histogram of the accuracy per class

correct = 0
total = 0
class_correct = list(0. for i in range(n_classes))
class_total = list(0. for i in range(n_classes))
with torch.no_grad():
    for data in test_dataloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(len(labels)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the test images: {100 * correct / total}%')

class_accuracies = []
for i in range(n_classes):
    accuracy = 100 * class_correct[i] / class_total[i]
    class_accuracies.append(accuracy)
    print(f'Accuracy of {test_dataset.classes[i]}: {accuracy}%')

#Plot the histogram of accuracies per class
plt.figure(figsize=(10,6))
plt.bar(range(n_classes), class_accuracies, align='center')
plt.xticks(range(n_classes), test_dataset.classes)
plt.xlabel('Class')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy per Class')
plt.show()